# 胶质瘤 scRNA-seq 整合分析 — GSE162631 + GSE273274

| 数据集 | 患者 | 样本数 | 组织类型 | 平台 |
|--------|------|--------|----------|------|
| GSE162631 | R1–R4 | 8 | Tumor + Peritumoral | 10X |
| GSE273274 | R1–R3 | 6 | GBM centre + surround | 10X |

流程: 加载 → QC → Doublet → 合并 → 归一化 → HVG → PCA → Harmony → Leiden → UMAP → 保存

In [ ]:
import os, re, gc, warnings
from pathlib import Path

import anndata
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor="white")
print(f"scanpy {sc.__version__}  anndata {anndata.__version__}")

In [ ]:
# QC 参数
min_cells    = 3
min_umi      = 1000
min_genes    = 200
max_genes    = 8000
mt_cap       = 10

# 分析参数
hvg_n_top    = 2000
pca_n_comps  = 50
batch_key    = "sample"
leiden_res_list = [0.5, 0.8, 1.0]
leiden_default  = 0.5

# 路径
base      = Path(".").resolve()
write_dir = base / "write" / "0_integrate_two_datasets"
fig_dir   = base / "figures" / "0_integrate_two_datasets"
os.makedirs(write_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

print(f"工作目录: {base}")
print(f"输出目录: {write_dir}")

## 1. 10X 数据读取

In [ ]:
def load_10x_sample(path, dataset_key, sample_name, prefix=""):
    """读取单个 10X 样本"""
    features_file = path / f"{prefix}features.tsv.gz"
    genes_file    = path / f"{prefix}genes.tsv.gz"
    created_symlink = False
    if not features_file.exists() and genes_file.exists():
        features_file.symlink_to(genes_file.name)
        created_symlink = True

    try:
        adata = sc.read_10x_mtx(path, var_names="gene_symbols",
                                make_unique=True, prefix=prefix, cache=False)
    finally:
        if created_symlink and features_file.is_symlink():
            features_file.unlink()

    if not np.issubdtype(adata.X.dtype, np.integer):
        adata.X = adata.X.astype(np.int32)

    adata.obs_names = [f"{dataset_key}_{sample_name}_{bc}" for bc in adata.obs_names]
    adata.obs["dataset"] = dataset_key
    adata.obs["sample"]  = sample_name
    return adata

## 2. 加载 GSE162631

In [ ]:
def load_gse162631(dataset_path, dataset_key, sample_anno_path):
    adatas = []
    tissue_map = {"T": "Tumor", "N": "Peritumoral"}
    sample_anno = pd.read_csv(sample_anno_path, index_col=0)

    sample_folders = sorted(
        d for d in dataset_path.iterdir()
        if d.is_dir() and not d.name.startswith(('.', '__'))
    )

    for folder in sample_folders:
        parts = folder.name.strip().split('_')
        if len(parts) < 2:
            print(f"  ⚠️ 跳过: {folder.name}")
            continue

        patient_id, tissue_suffix = parts[0], parts[1]
        tissue_type = tissue_map.get(tissue_suffix, "Unknown")

        # 定位 10X 三件套
        if (folder / "matrix.mtx.gz").exists():
            data_dir = folder
        elif (folder / folder.name / "matrix.mtx.gz").exists():
            data_dir = folder / folder.name
        else:
            candidates = list(folder.rglob("matrix.mtx.gz"))
            if candidates:
                data_dir = candidates[0].parent
            else:
                print(f"  ⚠️ 跳过 {folder.name}: 找不到 matrix.mtx.gz")
                continue

        # 匹配 GSM 编号
        tissue_synonyms = {"Peritumoral": "Peritumoral|Peripheral"}
        tissue_pattern = tissue_synonyms.get(tissue_type, tissue_type)
        matched = sample_anno[
            sample_anno['title'].str.contains(patient_id, case=False, na=False)
            & sample_anno['tissue'].str.contains(tissue_pattern, case=False, na=False)
        ]
        gsm_id = matched.iloc[0]['geo_accession'] if len(matched) > 0 else "Unknown"

        try:
            adata = load_10x_sample(data_dir, dataset_key, gsm_id)
            adata.obs["tissue_type"] = tissue_type
            adata.obs["patient_id"]  = f"{dataset_key}_{patient_id}"
            adata.obs["gsm_id"]      = gsm_id
            print(f"  ✅ {gsm_id} ({patient_id}, {tissue_type}): {adata.n_obs:,} × {adata.n_vars:,}")
            adatas.append(adata)
        except Exception as e:
            print(f"  ❌ {folder.name}: {e}")

    return adatas

adatas_162631 = load_gse162631(base / "GSE162631", "GSE162631", base / "GSE162631" / "GSE162631.csv")
print(f"→ {len(adatas_162631)} 个样本")

## 3. 加载 GSE273274

In [ ]:
def load_gse273274(dataset_path, dataset_key, sample_anno_path):
    adatas = []
    tissue_map = {"GBM centre": "Tumor", "GBM surround": "Peritumoral"}
    sample_anno = pd.read_csv(sample_anno_path)

    for _, row in sample_anno.iterrows():
        gsm_id = row['geo_accession']
        tissue_type = tissue_map.get(row['tissue:ch1'])
        if tissue_type is None:
            continue

        rep_match = re.search(r'replicate?d?\s*(\d+)', row['title'])
        patient_id = f"R{rep_match.group(1)}" if rep_match else "Unknown"

        mtx_files = list(dataset_path.glob(f"{gsm_id}_*.matrix.mtx.gz"))
        if not mtx_files:
            print(f"  ⚠️ 找不到 {gsm_id} 矩阵文件")
            continue
        prefix = mtx_files[0].name.replace("matrix.mtx.gz", "")

        try:
            adata = load_10x_sample(dataset_path, dataset_key, gsm_id, prefix=prefix)
            adata.obs["tissue_type"] = tissue_type
            adata.obs["patient_id"]  = f"{dataset_key}_{patient_id}"
            adata.obs["gsm_id"]      = gsm_id
            print(f"  ✅ {gsm_id} ({patient_id}, {tissue_type}): {adata.n_obs:,} × {adata.n_vars:,}")
            adatas.append(adata)
        except Exception as e:
            print(f"  ❌ {gsm_id}: {e}")

    return adatas

adatas_273274 = load_gse273274(base / "GSE273274", "GSE273274", base / "GSE273274" / "GSE273274.csv")
print(f"→ {len(adatas_273274)} 个样本")

In [ ]:
all_adatas = adatas_162631 + adatas_273274
print(f"共加载 {len(all_adatas)} 个样本")
for ds in ["GSE162631", "GSE273274"]:
    n = sum(1 for a in all_adatas if a.obs['dataset'].iat[0] == ds)
    print(f"  {ds}: {n} 个样本")

## 4. 质量控制

In [ ]:
def annotate_qc(adata):
    adata.var["mt"]   = adata.var_names.str.upper().str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.match(r'^RP[SL]\d')
    adata.var["hb"]   = adata.var_names.str.match(r'^HB[^P]')
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"],
                               percent_top=None, log1p=False, inplace=True)

def apply_qc(adata):
    adata = adata.copy()
    annotate_qc(adata)
    keep = (
        (adata.obs["total_counts"]      >= min_umi)
        & (adata.obs["n_genes_by_counts"] >= min_genes)
        & (adata.obs["n_genes_by_counts"] <= max_genes)
        & (adata.obs["pct_counts_mt"]     <= mt_cap)
    )
    n_rm = (~keep).sum()
    print(f"    移除 {n_rm} 个细胞 "
          f"(UMI<{min_umi}: {(adata.obs['total_counts'] < min_umi).sum()}, "
          f"genes<{min_genes}: {(adata.obs['n_genes_by_counts'] < min_genes).sum()}, "
          f"genes>{max_genes}: {(adata.obs['n_genes_by_counts'] > max_genes).sum()}, "
          f"MT>{mt_cap}%: {(adata.obs['pct_counts_mt'] > mt_cap).sum()})")
    return adata[keep].copy()

In [ ]:
# 标注 pre-QC 指标
for adata in all_adatas:
    annotate_qc(adata)

# 执行 QC
filtered_adatas = []
qc_rows = []

for adata in all_adatas:
    ds   = adata.obs["dataset"].iat[0]
    samp = adata.obs["sample"].iat[0]
    tt   = adata.obs["tissue_type"].iat[0]
    n_before = adata.n_obs

    adata_f = apply_qc(adata)
    n_after = adata_f.n_obs

    print(f"  {ds}/{samp} ({tt}): {n_before:,} → {n_after:,} ({n_after/max(n_before,1)*100:.1f}%)")
    filtered_adatas.append(adata_f)
    qc_rows.append(dict(dataset=ds, sample=samp, tissue_type=tt,
                        n_before=n_before, n_after=n_after))

qc_stats = pd.DataFrame(qc_rows)
total = qc_stats['n_after'].sum()
print(f"\n过滤后: {len(filtered_adatas)} 个样本, {total:,} 个细胞")

In [ ]:
# QC 可视化
qc_stats['retention'] = qc_stats['n_after'] / qc_stats['n_before']
qc_sorted = qc_stats.sort_values(['dataset', 'sample']).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(qc_sorted))
axes[0].bar(x, qc_sorted['n_before'], label='Before QC', alpha=0.7, color='#95a5a6')
axes[0].bar(x, qc_sorted['n_after'],  label='After QC',  alpha=0.8, color='#2ecc71')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(qc_sorted['sample'], rotation=45, ha='right', fontsize=8)
axes[0].set_title('Cell Counts Before vs After QC')
axes[0].set_ylabel('Number of Cells')
axes[0].legend()

colors = ['#e74c3c' if r < 0.5 else '#2ecc71' for r in qc_sorted['retention']]
axes[1].bar(x, qc_sorted['retention'], color=colors)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(qc_sorted['sample'], rotation=45, ha='right', fontsize=8)
axes[1].set_ylim(0, 1)
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.4, label='50%')
axes[1].set_title('Retention Rate per Sample')
axes[1].set_ylabel('Retention Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig(fig_dir / 'qc_retention_before_after.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# QC violin 对比
qc_metrics = ['pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb',
              'n_genes_by_counts', 'total_counts']

frames = []
for status, adatas_list in [('Before', all_adatas), ('After', filtered_adatas)]:
    for ad in adatas_list:
        df = ad.obs[qc_metrics].copy()
        df['status'] = status
        df['sample'] = ad.obs['sample'].iat[0]
        frames.append(df)
qc_df = pd.concat(frames, ignore_index=True)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, metric in zip(axes.flat[:5], qc_metrics):
    sns.violinplot(data=qc_df, x='status', y=metric, ax=ax, cut=0,
                   order=['Before', 'After'], inner='box')
    ax.set_title(metric, fontsize=10)
    ax.set_xlabel('')
axes.flat[-1].axis('off')

plt.suptitle('QC Metrics Before vs After Filtering', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(fig_dir / 'qc_violin_before_after.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 5. Doublet 检测 (Scrublet)

In [ ]:
doublet_stats = []
doublet_filtered = []

# 合理 doublet rate 范围: 0.5% ~ 15%
MIN_RATE, MAX_RATE = 0.005, 0.15

for adata in filtered_adatas:
    samp = adata.obs["sample"].iat[0]
    ds   = adata.obs["dataset"].iat[0]
    n_before = adata.n_obs

    expected_rate = min(max(n_before * 0.008 / 1000, 0.02), 0.15)
    try:
        sc.pp.scrublet(adata, expected_doublet_rate=expected_rate, random_state=42)
        scores = adata.obs["doublet_score"].values
        auto_rate = adata.obs["predicted_doublet"].mean()

        # 自动阈值结果不合理 → 用 expected_rate 百分位阈值覆盖
        if auto_rate > MAX_RATE or auto_rate < MIN_RATE:
            thr = np.percentile(scores, (1 - expected_rate) * 100)
            adata.obs["predicted_doublet"] = scores > thr
            new_rate = adata.obs["predicted_doublet"].mean()
            reason = "过高" if auto_rate > MAX_RATE else "过低"
            print(f"    ⚠️ {samp}: auto rate {auto_rate:.1%} {reason}, "
                  f"改用 percentile threshold={thr:.3f} → {new_rate:.1%}")
    except Exception as e:
        print(f"  ⚠️ {ds}/{samp} Scrublet 失败: {e}")
        adata.obs["predicted_doublet"] = False
        adata.obs["doublet_score"] = 0.0

    n_doublets = int(adata.obs["predicted_doublet"].sum())
    rate = n_doublets / n_before * 100
    adata_singlet = adata[~adata.obs["predicted_doublet"]].copy()

    print(f"  {ds}/{samp}: {n_before:,} → {adata_singlet.n_obs:,} (doublets: {n_doublets}, {rate:.1f}%)")
    doublet_filtered.append(adata_singlet)
    doublet_stats.append(dict(dataset=ds, sample=samp, n_before=n_before,
                              n_doublets=n_doublets, rate=rate, n_after=adata_singlet.n_obs))

doublet_df = pd.DataFrame(doublet_stats)
filtered_adatas = doublet_filtered

print(f"\n总 doublet: {doublet_df['n_doublets'].sum():,} / {doublet_df['n_before'].sum():,}")
print(f"平均 doublet 比例: {doublet_df['rate'].mean():.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df_s = doublet_df.sort_values("rate", ascending=True)
c = ['#e74c3c' if r > 10 else '#4C72B0' for r in df_s['rate']]
axes[0].barh(range(len(df_s)), df_s['rate'], color=c)
axes[0].set_yticks(range(len(df_s)))
axes[0].set_yticklabels(df_s['sample'], fontsize=8)
axes[0].set_xlabel('Doublet Rate (%)')
axes[0].set_title('Doublet Rate per Sample')
axes[0].axvline(x=10, color='red', linestyle='--', alpha=0.5, label='10%')
axes[0].legend(fontsize=8)

x = range(len(doublet_df))
axes[1].bar(x, doublet_df['n_before'], label='Before', alpha=0.7, color='#95a5a6')
axes[1].bar(x, doublet_df['n_after'],  label='After',  alpha=0.8, color='#2ecc71')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(doublet_df['sample'], rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Number of Cells')
axes[1].set_title('Cells Before/After Doublet Removal')
axes[1].legend(fontsize=8)

all_scores = pd.concat([a.obs[['doublet_score', 'sample']] for a in filtered_adatas], ignore_index=True)
axes[2].hist(all_scores['doublet_score'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[2].axvline(x=0.25, color='red', linestyle='--', label='threshold=0.25')
axes[2].set_xlabel('Doublet Score')
axes[2].set_ylabel('Count')
axes[2].set_title('Doublet Score Distribution (singlets)')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(fig_dir / 'doublet_detection_summary.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 6. 合并

In [ ]:
adata_combined = sc.concat(filtered_adatas, join="inner", merge="unique")
gc.collect()
print(f"合并: {adata_combined.n_obs:,} cells × {adata_combined.n_vars:,} genes")

# 基因过滤
n_pre = adata_combined.n_vars
sc.pp.filter_genes(adata_combined, min_cells=min_cells)
print(f"基因过滤: {n_pre:,} → {adata_combined.n_vars:,}")

print(f"\n样本分布:\n{adata_combined.obs['sample'].value_counts().to_string()}")
print(f"\n数据集分布:\n{adata_combined.obs['dataset'].value_counts().to_string()}")

# 患者配对
print("\n患者配对:")
for pid in sorted(adata_combined.obs['patient_id'].unique()):
    mask = adata_combined.obs['patient_id'] == pid
    tissues = sorted(adata_combined.obs.loc[mask, 'tissue_type'].unique())
    print(f"  {pid}: {mask.sum():,} cells, {tissues}")

## 7. 归一化与特征选择

In [ ]:
# 保存原始 counts
adata_combined.layers["counts"] = adata_combined.X.copy()

sc.pp.normalize_total(adata_combined, target_sum=1e4)
sc.pp.log1p(adata_combined)
print("✅ normalize_total(1e4) + log1p")

In [ ]:
sc.pp.highly_variable_genes(
    adata_combined, n_top_genes=hvg_n_top,
    flavor="seurat_v3", batch_key=batch_key, layer="counts",
)
n_hvg = adata_combined.var['highly_variable'].sum()
print(f"✅ HVG: {n_hvg} / {adata_combined.n_vars}")

sc.pl.highly_variable_genes(adata_combined, show=False)
plt.gcf().set_size_inches(8, 5)
plt.suptitle(f'Highly Variable Genes (n={n_hvg})', fontsize=12)
plt.tight_layout()
plt.savefig(fig_dir / 'hvg_selection.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 8. Harmony 批次校正

In [ ]:
import harmonypy as hm

# HVG 子集 → regress → scale → PCA
adata_hvg = adata_combined[:, adata_combined.var["highly_variable"]].copy()

regress_keys = ['total_counts', 'pct_counts_mt']
regress_present = [k for k in regress_keys if k in adata_hvg.obs.columns]
print(f"Regressing out: {regress_present}")
sc.pp.regress_out(adata_hvg, regress_present)

sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=pca_n_comps, random_state=42)
adata_combined.obsm["X_pca"] = adata_hvg.obsm["X_pca"]
adata_combined.uns["pca"] = adata_hvg.uns["pca"]
del adata_hvg; gc.collect()
print(f"✅ PCA: {pca_n_comps} 个主成分")

In [ ]:
sc.pl.pca_variance_ratio(adata_combined, n_pcs=pca_n_comps, log=True)

In [ ]:
# 校正前 UMAP
sc.pp.neighbors(adata_combined, use_rep="X_pca", key_added="pre_harmony", random_state=42)
sc.tl.umap(adata_combined, neighbors_key="pre_harmony", random_state=42)
umap_pre = adata_combined.obsm["X_umap"].copy()

# Harmony
n_batches = adata_combined.obs[batch_key].nunique()
print(f"Harmony (batch_key='{batch_key}', {n_batches} batches)...")
ho = hm.run_harmony(adata_combined.obsm["X_pca"], adata_combined.obs, batch_key, random_state=42)
adata_combined.obsm["X_pca_harmony"] = ho.Z_corr.T
print("✅ Harmony 完成")

# 校正后 UMAP
sc.pp.neighbors(adata_combined, use_rep="X_pca_harmony", random_state=42)
sc.tl.umap(adata_combined, random_state=42)
print("✅ 校正后 UMAP 完成")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 校正前
for samp in adata_combined.obs[batch_key].unique():
    mask = adata_combined.obs[batch_key] == samp
    axes[0, 0].scatter(umap_pre[mask, 0], umap_pre[mask, 1], s=1, alpha=0.5, label=samp)
axes[0, 0].set_title('Before Harmony — by Sample')
axes[0, 0].legend(markerscale=5, fontsize=6, loc='best')

for tt, color in [('Tumor', '#e74c3c'), ('Peritumoral', '#3498db')]:
    mask = adata_combined.obs['tissue_type'] == tt
    axes[0, 1].scatter(umap_pre[mask, 0], umap_pre[mask, 1], s=1, alpha=0.3, c=color, label=tt)
axes[0, 1].set_title('Before Harmony — by Tissue Type')
axes[0, 1].legend(markerscale=5)

# 校正后
sc.pl.umap(adata_combined, color=batch_key, ax=axes[1, 0], show=False,
           title='After Harmony — by Sample', s=3)
sc.pl.umap(adata_combined, color='tissue_type', ax=axes[1, 1], show=False,
           title='After Harmony — by Tissue Type', s=3)

plt.tight_layout()
plt.savefig(fig_dir / 'harmony_before_after.pdf', dpi=300, bbox_inches='tight')
plt.show()

del umap_pre, ho; gc.collect()

## 9. Leiden 聚类

In [ ]:
for res in leiden_res_list:
    key = f"leiden_res_{res}"
    sc.tl.leiden(adata_combined, resolution=res, key_added=key,
                random_state=42, flavor="igraph", n_iterations=2)
    n_cl = adata_combined.obs[key].nunique()
    print(f"  res={res}: {n_cl} clusters")

adata_combined.obs["leiden"] = adata_combined.obs[f"leiden_res_{leiden_default}"]
print(f"\n默认 leiden (res={leiden_default}): {adata_combined.obs['leiden'].nunique()} clusters")

In [ ]:
sc.pl.umap(adata_combined, color=[f"leiden_res_{r}" for r in leiden_res_list],
           ncols=2, legend_loc="on data", frameon=False)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

sc.pl.umap(adata_combined, color="leiden", legend_loc="on data",
           title="Leiden Clusters", ax=axes[0, 0], show=False, legend_fontoutline=2)
sc.pl.umap(adata_combined, color="sample", legend_loc="right margin",
           title="Samples", ax=axes[0, 1], show=False, s=5)
sc.pl.umap(adata_combined, color="tissue_type", legend_loc="right margin",
           title="Tissue Type", ax=axes[1, 0], show=False,
           palette={'Tumor': '#e74c3c', 'Peritumoral': '#3498db'})
sc.pl.umap(adata_combined, color="dataset", legend_loc="right margin",
           title="Dataset", ax=axes[1, 1], show=False)

plt.tight_layout()
plt.savefig(fig_dir / "umap_integrated_overview.pdf", dpi=300, bbox_inches="tight")
plt.show()

## 10. 保存

In [ ]:
out_path = write_dir / "adata1.h5ad"
adata_combined.write(out_path)
fsize = out_path.stat().st_size / 1024**3
print(f"✅ 已保存: {out_path}  ({fsize:.2f} GB)")
print(f"   {adata_combined.n_obs:,} cells × {adata_combined.n_vars:,} genes")
print(f"   layers: {list(adata_combined.layers.keys())}")
print(f"   obsm:   {list(adata_combined.obsm.keys())}")
print(f"   raw:    {'✅' if adata_combined.raw is not None else '❌'}")